<a href="https://colab.research.google.com/github/prasad26112006/sivaprasad/blob/siva/cluster_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import precision_score, recall_score, f1_score
from scipy.stats import mode
from collections import Counter


categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc"
]

data = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes")
)

documents = data.data
true_labels = data.target
class_names = data.target_names

print("Number of documents:", len(documents))
print("Number of classes:", len(class_names))
print("Classes:", class_names)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    min_df=2,
    max_df=0.95
)

X = vectorizer.fit_transform(documents)

print("TF-IDF matrix shape:", X.shape)


k = len(class_names)

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X)


def cluster_to_class_mapping(y_true, y_cluster):
    mapping = {}

    for cluster in np.unique(y_cluster):
        indices = np.where(y_cluster == cluster)[0]

        if len(indices) > 0:
            most_common = Counter(y_true[indices]).most_common(1)[0][0]
            mapping[cluster] = most_common

    return np.array([mapping[c] for c in y_cluster]), mapping


predicted_labels, mapping = cluster_to_class_mapping(
    true_labels,
    cluster_labels
)


def purity_score(y_true, y_pred):
    contingency_matrix = np.zeros(
        (len(np.unique(y_true)), len(np.unique(y_pred))),
        dtype=int
    )

    for i, true_class in enumerate(np.unique(y_true)):
        for j, cluster in enumerate(np.unique(y_pred)):
            contingency_matrix[i, j] = np.sum(
                (y_true == true_class) & (y_pred == cluster)
            )

    return np.sum(np.max(contingency_matrix, axis=0)) / len(y_true)


purity = purity_score(true_labels, cluster_labels)

precision = precision_score(
    true_labels,
    predicted_labels,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    true_labels,
    predicted_labels,
    average="weighted",
    zero_division=0
)

f_measure = f1_score(
    true_labels,
    predicted_labels,
    average="weighted",
    zero_division=0
)


print("\n========== PERFORMANCE MEASURES ==========")
print(f"Purity    : {purity:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F-measure : {f_measure:.4f}")


print("\n========== CLUSTER MAPPING ==========")

for cluster, class_id in mapping.items():
    print(
        f"Cluster {cluster} -> "
        f"{class_names[class_id]}"
    )


print("\n========== CLUSTER SIZES ==========")

for cluster in range(k):
    count = np.sum(cluster_labels == cluster)
    print(f"Cluster {cluster}: {count} documents")


print("\n========== SAMPLE CLUSTER RESULTS ==========")

for i in range(min(10, len(documents))):
    print(
        f"\nDocument {i + 1}"
        f"\nActual Class   : {class_names[true_labels[i]]}"
        f"\nPredicted Class: {class_names[predicted_labels[i]]}"
        f"\nCluster        : {cluster_labels[i]}"
    )

Number of documents: 3729
Number of classes: 4
Classes: ['comp.graphics', 'rec.sport.baseball', 'sci.space', 'talk.politics.misc']
TF-IDF matrix shape: (3729, 5000)

========== PERFORMANCE MEASURES ==========
Purity    : 0.6213
Precision : 0.8334
Recall    : 0.6213
F-measure : 0.6402

========== CLUSTER MAPPING ==========
Cluster 0 -> rec.sport.baseball
Cluster 1 -> sci.space
Cluster 2 -> talk.politics.misc
Cluster 3 -> comp.graphics

========== CLUSTER SIZES ==========
Cluster 0: 574 documents
Cluster 1: 372 documents
Cluster 2: 2095 documents
Cluster 3: 688 documents

========== SAMPLE CLUSTER RESULTS ==========

Document 1
Actual Class   : rec.sport.baseball
Predicted Class: rec.sport.baseball
Cluster        : 0

Document 2
Actual Class   : talk.politics.misc
Predicted Class: talk.politics.misc
Cluster        : 2

Document 3
Actual Class   : talk.politics.misc
Predicted Class: talk.politics.misc
Cluster        : 2

Document 4
Actual Class   : talk.politics.misc
Predicted Class: talk